## **HR EDA 코드 설명**
- 목적: HR Data의 통계적 가설 검정을 위한 코드
- 사용된 데이터: 임직원의 특징과 이직 여부를 포함한 데이터 (*이직 구분자: Attrition)

In [2]:
import pandas as pd
from sqlalchemy import create_engine, text
from scipy.stats import chi2_contingency, ttest_ind

# Localhost MySQL 연결 
USER = "root"
PASSWORD = ""
HOST = "127.0.0.1"
PORT = "3306"
DB = "hr"

engine = create_engine(f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB}")

In [3]:
# SQL 결과 확인 함수
def run_query(query):
    return pd.read_sql(text(query), engine)

## 가설1. 이직자의 평균 급여는 더 낮은가?

In [5]:
# 이직 여부와 급여만 가져오는 쿼리
query_salary = """
SELECT 
    Attrition,
    MonthlyIncome
FROM hr_employee_attrition
WHERE MonthlyIncome IS NOT NULL
  AND Attrition IS NOT NULL;
"""
df_salary = run_query(query_salary)
df_salary.head()

,Attrition,MonthlyIncome
0,Yes,5993
1,No,5130
2,Yes,2090
3,No,2909
4,No,3468


In [6]:
# 이직한 사람의 급여 정보 추출 
yes_income = df_salary[df_salary["Attrition"] == "Yes"]["MonthlyIncome"]

# 잔류한 사람의 급여 정보 추출 
no_income = df_salary[df_salary["Attrition"] == "No"]["MonthlyIncome"]

print("이직자의 평균 급여:", round(yes_income.mean(), 1))
print("잔류자의 평균 급여:", round(no_income.mean(), 1))

이직자의 평균 급여: 4787.1
잔류자의 평균 급여: 6832.7


In [7]:
# 낮은 급여가 이직 여부에 영향을 미치는지를 T-Test로 검정
t_stat, p_salary = ttest_ind(yes_income, no_income, equal_var=False)

print("=== MonthlyIncome by Attrition ===")
print("t-statistic:", t_stat)
print("p-value:", p_salary)

# 가설 검정 해설
if p_salary <= 0.05:
    print('낮은 급여가 이직 여부에 영향을 미치는 가설은 유의미하다.')
else:
    print('가설은 무의미하다')
 

=== MonthlyIncome by Attrition ===
t-statistic: -7.482621586644742
p-value: 4.433588628286071e-13
낮은 급여가 이직 여부에 영향을 미치는 가설은 유의미하다.


In [8]:
# 이직 여부에 따른 급여 요약 표 
query_salary_summary = """
SELECT
    Attrition,
    COUNT(*) AS total_count,
    ROUND(AVG(MonthlyIncome), 2) AS avg_monthly_income,
    MIN(MonthlyIncome) AS min_income,
    MAX(MonthlyIncome) AS max_income
FROM hr_employee_attrition
GROUP BY Attrition;
"""
salary_summary = run_query(query_salary_summary)
salary_summary

,Attrition,total_count,avg_monthly_income,min_income,max_income
0,Yes,237,4787.09,1009,19859
1,No,1233,6832.74,1051,19999


## 가설2. 저연차 직원의 이직률이 유의미하게 높은가?

In [10]:
query_low_level = """
SELECT 
    JobLevel,
    Attrition
FROM hr_employee_attrition
WHERE JobLevel IS NOT NULL
  AND Attrition IS NOT NULL;
"""
df_low_level = run_query(query_low_level)
df_low_level.head()

,JobLevel,Attrition
0,2,Yes
1,2,No
2,1,Yes
3,1,No
4,1,No


In [11]:
# 교차 검증을 위한 표 생성 
contingency_low_level = pd.crosstab(df_low_level["JobLevel"], df_low_level["Attrition"])
contingency_low_level

Attrition,No,Yes
JobLevel,,
1,400,143
2,482,52
3,186,32
4,101,5
5,64,5


In [12]:
# 저년차 레벨의 직원이 이직률이 높다는 것을 카이제곱 테스트로 검정
chi2_low_level, p_low_level, dof_low_level, expected_low_level = chi2_contingency(contingency_low_level)

print("=== Early vs NonEarly Attrition ===")
print("chi2:", chi2_low_level)
print("p-value:", p_low_level)

# 가설 검정 해설
if p_low_level <= 0.05:
    print('저연차 레벨의 직원의 이직률이 높다는 가설은 유의미하다.')
else:
    print('가설은 무의미하다')

=== Early vs NonEarly Attrition ===
chi2: 72.5290131066739
p-value: 6.634684715458957e-15
저연차 레벨의 직원의 이직률이 높다는 가설은 유의미하다.


In [13]:
# 직급 레벨 별 이직률 요약 표
query_low_level_rate = """
SELECT
    JobLevel,
    COUNT(*) AS total_count,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS attrition_count,
    ROUND(100.0 * SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS attrition_rate_pct
FROM hr_employee_attrition
GROUP BY JobLevel
ORDER BY JobLevel;
"""
low_level_rate = run_query(query_low_level_rate)
low_level_rate

,JobLevel,total_count,attrition_count,attrition_rate_pct
0,1,543,143.0,26.34
1,2,534,52.0,9.74
2,3,218,32.0,14.68
3,4,106,5.0,4.72
4,5,69,5.0,7.25


## 가설3. 초과근무가 있는 직원이 이직률이 더 높은가?

In [15]:
query_ot = """
SELECT 
    OverTime,
    Attrition
FROM hr_employee_attrition
WHERE OverTime IS NOT NULL
  AND Attrition IS NOT NULL;
"""
df_ot = run_query(query_ot)
df_ot.head()

,OverTime,Attrition
0,Yes,Yes
1,No,No
2,Yes,Yes
3,Yes,No
4,No,No


In [16]:
contingency_ot = pd.crosstab(df_ot["OverTime"], df_ot["Attrition"])
contingency_ot

Attrition,No,Yes
OverTime,,
No,944,110
Yes,289,127


In [17]:
# 초과 근무가 있는 직원이 이직률이 높다는 것을 카이제곱 테스트로 검정
chi2_ot, p_ot, dof_ot, expected_ot = chi2_contingency(contingency_ot)

print("=== Slide 7-2: OverTime vs Attrition ===")
print("chi2:", chi2_ot)
print("p-value:", p_ot)

# 가설 검정 해설
if p_ot <= 0.05:
    print('초과 근무가 있는 직원의 이직률이 높다는 가설은 유의미하다.')
else:
    print('가설은 무의미하다')

=== Slide 7-2: OverTime vs Attrition ===
chi2: 87.56429365828768
p-value: 8.15842372153832e-21
초과 근무가 있는 직원의 이직률이 높다는 가설은 유의미하다.


In [18]:
# 초과근무 별 이직률 요약 표
query_ot_rate = """
SELECT
    OverTime,
    COUNT(*) AS total_count,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS attrition_count,
    ROUND(100.0 * SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS attrition_rate_pct
FROM hr_employee_attrition
GROUP BY OverTime;
"""
ot_rate = run_query(query_ot_rate)
ot_rate

,OverTime,total_count,attrition_count,attrition_rate_pct
0,Yes,416,127.0,30.53
1,No,1054,110.0,10.44


## 가설4. 낮은 직무 만족도는 이직률이 높은가?

In [20]:
query_js = """
SELECT 
    JobSatisfaction,
    Attrition
FROM hr_employee_attrition
WHERE JobSatisfaction IS NOT NULL
  AND Attrition IS NOT NULL;
"""
df_js = run_query(query_js)
df_js.head()

,JobSatisfaction,Attrition
0,4,Yes
1,2,No
2,3,Yes
3,3,No
4,2,No


In [21]:
contingency_js = pd.crosstab(df_js["JobSatisfaction"], df_js["Attrition"])
contingency_js

Attrition,No,Yes
JobSatisfaction,,
1,223,66
2,234,46
3,369,73
4,407,52


In [22]:
# 직업 만족도가 낮은 직원이 이직률이 높다는 것을 카이제곱 테스트로 검정
chi2_js, p_js, dof_js, expected_js = chi2_contingency(contingency_js)

print("=== JobSatisfaction vs Attrition ===")
print("chi2:", chi2_js)
print("p-value:", p_js)

# 가설 검정 해설
if p_js <= 0.05:
    print('직업 만족도가 낮은 직원의 이직률이 높다는 가설은 유의미하다.')
else:
    print('가설은 무의미하다')

=== JobSatisfaction vs Attrition ===
chi2: 17.505077010348
p-value: 0.0005563004510387556
직업 만족도가 낮은 직원의 이직률이 높다는 가설은 유의미하다.


In [23]:
# 직업 만족도 별 이직률 요약 표
query_js_rate = """
SELECT
    JobSatisfaction,
    COUNT(*) AS total_count,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS attrition_count,
    ROUND(100.0 * SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS attrition_rate_pct
FROM hr_employee_attrition
GROUP BY JobSatisfaction;
"""
js_rate = run_query(query_js_rate)
js_rate

,JobSatisfaction,total_count,attrition_count,attrition_rate_pct
0,4,459,52.0,11.33
1,2,280,46.0,16.43
2,3,442,73.0,16.52
3,1,289,66.0,22.84


## OverTime과 낮은 만족도 결합 상태의 이직률 확인

In [25]:
query_ot_js = """
SELECT 
    OverTime,
    JobSatisfaction,
    Attrition
FROM hr_employee_attrition
WHERE OverTime IS NOT NULL
  AND JobSatisfaction IS NOT NULL
  AND Attrition IS NOT NULL;
"""
df_ot_js = run_query(query_ot_js)
df_ot_js.head()

,OverTime,JobSatisfaction,Attrition
0,Yes,4,Yes
1,No,2,No
2,Yes,3,Yes
3,Yes,3,No
4,No,2,No


In [26]:
# 직업 만족도와 초과 근무를 결합한 컬럼 생성 
df_ot_js["OT_Sat"] = df_ot_js["OverTime"] + "_Sat" + df_ot_js["JobSatisfaction"].astype(str)
df_ot_js[["OverTime", "JobSatisfaction", "OT_Sat", "Attrition"]].head()

,OverTime,JobSatisfaction,OT_Sat,Attrition
0,Yes,4,Yes_Sat4,Yes
1,No,2,No_Sat2,No
2,Yes,3,Yes_Sat3,Yes
3,Yes,3,Yes_Sat3,No
4,No,2,No_Sat2,No


In [27]:
contingency_ot_js = pd.crosstab(df_ot_js["OT_Sat"], df_ot_js["Attrition"])
contingency_ot_js

Attrition,No,Yes
OT_Sat,,
No_Sat1,169,36
No_Sat2,191,20
No_Sat3,289,32
No_Sat4,295,22
Yes_Sat1,54,30
Yes_Sat2,43,26
Yes_Sat3,80,41
Yes_Sat4,112,30


In [28]:
# 초과 근무가 있으면서 직업 만족도가 낮은 직원이 이직률이 높다는 것을 카이제곱 테스트로 검정
chi2_ot_js, p_ot_js, dof_ot_js, expected_ot_js = chi2_contingency(contingency_ot_js)

print("=== OverTime + JobSatisfaction vs Attrition ===")
print("chi2:", chi2_ot_js)
print("p-value:", p_ot_js)

# 가설 검정 해설
if p_ot_js <= 0.05:
    print('초과 근무가 있으면서 직업 만족도가 낮은 직원의 이직률이 높다는 가설은 유의미하다.')
else:
    print('가설은 무의미하다')

=== OverTime + JobSatisfaction vs Attrition ===
chi2: 114.36938650817412
p-value: 1.1368175295171002e-21
초과 근무가 있으면서 직업 만족도가 낮은 직원의 이직률이 높다는 가설은 유의미하다.


In [30]:
ot_js_rate = contingency_ot_js.copy()
ot_js_rate["total_count"] = ot_js_rate["No"] + ot_js_rate["Yes"]
ot_js_rate["attrition_rate_pct"] = round(100 * ot_js_rate["Yes"] / ot_js_rate["total_count"], 2)
ot_js_rate = ot_js_rate.sort_values("attrition_rate_pct", ascending=False)
ot_js_rate

Attrition,No,Yes,total_count,attrition_rate_pct
OT_Sat,,,,
Yes_Sat2,43,26,69,37.68
Yes_Sat1,54,30,84,35.71
Yes_Sat3,80,41,121,33.88
Yes_Sat4,112,30,142,21.13
No_Sat1,169,36,205,17.56
No_Sat3,289,32,321,9.97
No_Sat2,191,20,211,9.48
No_Sat4,295,22,317,6.94


In [32]:
# 초과 근무 및 직업 만족도 별 이직률 요약 표
query_ot_js_rate = """
SELECT
    CONCAT(OverTime, '_Sat', JobSatisfaction) AS OT_Sat,
    COUNT(*) AS total_count,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS attrition_count,
    ROUND(100.0 * SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS attrition_rate_pct
FROM hr_employee_attrition
WHERE OverTime IS NOT NULL
  AND JobSatisfaction IS NOT NULL
  AND Attrition IS NOT NULL
GROUP BY CONCAT(OverTime, '_Sat', JobSatisfaction)
ORDER BY attrition_rate_pct DESC;
"""
run_query(query_ot_js_rate)

,OT_Sat,total_count,attrition_count,attrition_rate_pct
0,Yes_Sat2,69,26.0,37.68
1,Yes_Sat1,84,30.0,35.71
2,Yes_Sat3,121,41.0,33.88
3,Yes_Sat4,142,30.0,21.13
4,No_Sat1,205,36.0,17.56
5,No_Sat3,321,32.0,9.97
6,No_Sat2,211,20.0,9.48
7,No_Sat4,317,22.0,6.94
